In [1]:
%load_ext autoreload
%autoreload 2

# Extraction datasets inspection

Run and inspect every structured-extraction dataset used by the MixedDecoder training
(`s_03_11_train_mixed_decoder.py`), from `cite` through `sql`:

| type | module | task |
|------|--------|------|
| `cite` | `mllm.train.encdec_graph_bert` | masked-citation recall (anchor span) |
| `keyval` | `mllm.train.key_val_recall` | key → value recall |
| `jsonfield` | `mllm.train.json_field_recall` | JSON path → field value |
| `jsonata` | `mllm.train.jsonata_recall` | JSONata/jq selection + transform |
| `xmlxpath` | `mllm.train.xml_xpath_recall` | XML/XPath → node value |
| `sql` | `mllm.train.sql_select_recall` | SQL select/aggregate |

For each we build one batch, **decode the tokens** (encoder record, prompt, target),
and verify the target value is actually present in the encoded context (the
“needle-in-context” forcing function the datasets are designed around).

In [2]:
import sys
from typing import List, Optional
if '..' not in sys.path: sys.path.append('..')

import numpy as np
import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer

from mllm.train.encdec_graph_bert import MaskedCiteDataset, create_masked_cite_dataloader
from mllm.train.key_val_recall import KeyValRecallCfg, KeyValRecallDataset, create_key_val_recall_dataloader
from mllm.train.json_field_recall import JsonFieldRecallCfg, JsonFieldRecallDataset, create_json_field_recall_dataloader
from mllm.train.jsonata_recall import JsonataRecallCfg, JsonataRecallDataset, create_jsonata_recall_dataloader
from mllm.train.xml_xpath_recall import XmlXpathRecallCfg, XmlXpathRecallDataset, create_xml_xpath_recall_dataloader
from mllm.train.sql_select_recall import SqlSelectRecallCfg, SqlSelectRecallDataset, create_sql_select_recall_dataloader


## Config

We load the **real Wikipedia dataset** from the local HuggingFace cache
(`DATA_PATH/wikipedia/WIKI_DS_NAME`, here `20220301.en`) — the exact same corpus the
MixedDecoder training reads. To keep the notebook fast and reproducible we shuffle and
select a small slice of articles; the recall builders only need *real words / spans*
drawn from each document's `text`, so a few thousand articles exercise every code path.

Encoder and decoder tokenizer are both `bert-base-uncased` (shared vocab) so decoded
tokens line up exactly between the encoder record and the decoder prompt/target.


In [3]:
import os
from pathlib import Path

# Local HuggingFace cache that holds the real Wikipedia dataset
# (cache layout: DATA_PATH/wikipedia/WIKI_DS_NAME). Override DATA_PATH if needed.
DATA_PATH = Path(os.environ.get('DATA_PATH', 'Q:/data'))
WIKI_DS_NAME = '20220301.en'
N_DOCS = 5000          # how many shuffled articles to keep as the working corpus

device = torch.device('cpu')
max_seq_len = 192
batch_size = 5
n_special_toks = 1000
seed = 42

tkz_enc = AutoTokenizer.from_pretrained('bert-base-uncased')
tkz_dec = AutoTokenizer.from_pretrained('bert-base-uncased')
if tkz_dec.pad_token is None:
    tkz_dec.pad_token = tkz_dec.eos_token
print('enc vocab:', len(tkz_enc), '| dec vocab:', len(tkz_dec))
print('data path:', DATA_PATH, '| wiki:', WIKI_DS_NAME)


enc vocab: 30522 | dec vocab: 30522
data path: Q:\data | wiki: 20220301.en


In [4]:
# Load the real Wikipedia dataset from the local cache (same corpus as training).
# This matches `n_02_dswiki.ipynb`: load_dataset('wikipedia', WIKI_DS_NAME, cache_dir=DATA_PATH).
wiki_full = load_dataset('wikipedia', WIKI_DS_NAME, cache_dir=str(DATA_PATH))['train']
print('full wiki articles:', len(wiki_full))

# Shuffle + take a small working slice so the notebook is fast and reproducible.
ds = wiki_full.shuffle(seed=seed).select(range(min(N_DOCS, len(wiki_full))))
ds = ds.select_columns(['text']) if 'text' in ds.column_names else ds
print('corpus docs:', len(ds))
print('\n--- sample article (truncated) ---')
print(ds[0]['text'][:600], '...')


Using the latest cached version of the dataset since wikipedia couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration '20220301.en' at Q:\data\wikipedia\20220301.en\2.0.0\d41137e149b2ea90eead07e7e3f805119a8c22dd1d5b61651af8e3e3ee736001 (last modified on Sun Nov 16 11:14:09 2025).


Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

full wiki articles: 6458670
corpus docs: 5000

--- sample article (truncated) ---
William Edward Whitehouse (20 May 1859 – 12 January 1935) was an English cellist.

Career
He studied for one year with Alfredo Piatti, for whom he deputised (taking his place in concerts when called upon), and was his favourite pupil. He went on to teach at the Royal Academy of Music, Royal College of Music and King's College, Cambridge; his students included Felix Salmond and Beatrice Harrison, who both became closely associated with Edward Elgar. He played with violinist Joseph Joachim, and formed The London Trio with violinist Achille Simonetti and pianist Amina Goodwin. He edited Piatti's  ...


## Decoding helpers

Each dataset yields a `MaskedCiteBatch`. The tensors we care about:

* `inp_toks` / `inp_masked_toks` — the encoder record (**encoder** vocab).
* `prompts_toks` — the query the decoder is conditioned on (**decoder** vocab).
* `cites_toks` — the target the decoder must produce (**decoder** vocab).

Per-sample `tokens_subsets` also carry the raw `prompt` string plus the encoder-vocab
value (`toks_cite`) which we use to confirm the needle is inside the record.

In [5]:
def _decode_row(toks_row: torch.Tensor, att_row: Optional[torch.Tensor], tkz, skip_special=False) -> str:
    if att_row is not None:
        n = int(att_row.sum().item())
        ids = toks_row[:n].tolist()
    else:
        ids = toks_row.tolist()
    return tkz.decode(ids, skip_special_tokens=skip_special)


def _contains_subseq(haystack: List[int], needle: List[int]) -> bool:
    if not needle:
        return True
    n, m = len(haystack), len(needle)
    for i in range(n - m + 1):
        if haystack[i:i + m] == needle:
            return True
    return False


def show_batch(name: str, batch, n: int = 3):
    print('=' * 100)
    print(f'### {name}  |  batch_size={batch.inp_toks.shape[0]}  enc_len={batch.inp_toks.shape[1]}  '
          f'dec_prompt_len={batch.prompts_toks.shape[1]}  dec_cite_len={batch.cites_toks.shape[1]}')
    print('=' * 100)
    n = min(n, batch.inp_toks.shape[0])
    for i in range(n):
        sub = batch.tokens_subsets[i]
        record = _decode_row(batch.inp_toks[i], batch.inp_att_mask[i], tkz_enc)
        masked = _decode_row(batch.inp_masked_toks[i], batch.inp_att_mask[i], tkz_enc)
        prompt = _decode_row(batch.prompts_toks[i], batch.prompts_att_mask[i], tkz_dec)
        target = _decode_row(batch.cites_toks[i], batch.cites_att_mask[i], tkz_dec)
        # Needle-in-context check: encoder-vocab target value must live inside the record.
        needle_in_ctx = _contains_subseq(list(sub.toks_inp), list(sub.toks_cite))
        print(f'\n--- sample {i} ---')
        print(f'  ENC RECORD : {record}')
        if masked != record:
            print(f'  ENC MASKED : {masked}')
        print(f'  PROMPT     : {prompt}')
        print(f'  TARGET     : {target}')
        print(f'  needle in context: {needle_in_ctx}')

## 1. `cite` — masked citation recall

In [6]:
cite_ds = MaskedCiteDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    mask_cfg=None, prompt_all=False, device=device, tkz_dec=tkz_dec,
)
cite_ds_loader = create_masked_cite_dataloader(cite_ds, batch_size=batch_size)
cite_ds.shuffle(seed=seed)
cite_batch = next(cite_ds_loader)
cite_batch = next(cite_ds_loader)
show_batch('cite', cite_batch)

Token indices sequence length is longer than the specified maximum sequence length for this model (823 > 512). Running this sequence through the model will result in indexing errors


R0. Create MaskedCiteDataset dataloader. batch_size=5. shuffle=True.
### cite  |  batch_size=5  enc_len=192  dec_prompt_len=27  dec_cite_len=152

--- sample 0 ---
  ENC RECORD : [CLS] by the lake shore and michigan southern railway and had a freight house installed across the tracks in 1907. the building is constructed of red brick trimmed with limestone, which is used for the window surrounds and belt course bridgesilis totaled. the station was originally set amid a well - kept garden that displayed neat beds of colorful flowers and a row of trees along the tracks ; this manicured landscape was not only a pretty introduction to the city for first time visitors, but it also buffered the streets of downtown from the noise and dirt associated with steam engines and freight trains. the station and the railroad were acquired by the new york central railroad in 1914. nyc merged with nobel threworro the pennsylvania railroad in 1968, and passenger service was taken over by amtrak in 1971. th

## 2. `keyval` — key → value recall

In [7]:
keyval_cfg = KeyValRecallCfg(min_pairs=4, max_pairs=12, value_max_words=3)
keyval_ds = KeyValRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=keyval_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
keyval_ds_loader = create_key_val_recall_dataloader(keyval_ds, batch_size=batch_size)
keyval_ds.shuffle(seed=seed)
keyval_batch = next(keyval_ds_loader)
show_batch('keyval', keyval_batch)

R0. Create KeyValRecallDataset dataloader. batch_size=5.
### keyval  |  batch_size=5  enc_len=75  dec_prompt_len=12  dec_cite_len=4

--- sample 0 ---
  ENC RECORD : [CLS] edward = s. | ruthazer = ( born | in = 1966 in | new = york, | ny = ) is a | canadian = neuroscientist | and = professor at mcgill | university = in montreal, | quebec =. research ruthazer | research = utilizes in | vivo = multiphoton fluorescence microscopy [SEP]
  PROMPT     : key : " in ". retrieve its value.
  TARGET     : 1966 in [SEP]
  needle in context: True

--- sample 1 ---
  ENC RECORD : [CLS] choanephora - cucurbitarum is a ; fungal - plant pathogen that ; causes - fruit and blossom ; rot - of various ; cucurbits -. it can ; also - affect okra ; snap - bean, ; and - southern pea, ; may - cause a [SEP]
  PROMPT     : key : " causes ". retrieve its value.
  TARGET     : fruit and blossom [SEP]
  needle in context: True

--- sample 2 ---
  ENC RECORD : [CLS] francisco : escudero may refer ; to : : francisco e

## 3. `jsonfield` — JSON path → field value

In [8]:
jsonfield_cfg = JsonFieldRecallCfg(
    min_fields=4, max_fields=10, max_depth=3, max_array_len=4, value_max_words=3,
)
jsonfield_ds = JsonFieldRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=jsonfield_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
jsonfield_ds.shuffle(seed=seed)
jsonfield_batch = next(create_json_field_recall_dataloader(jsonfield_ds, batch_size=batch_size))
show_batch('jsonfield', jsonfield_batch)

R0. Create JsonFieldRecallDataset dataloader. batch_size=5.
### jsonfield  |  batch_size=5  enc_len=115  dec_prompt_len=20  dec_cite_len=7

--- sample 0 ---
  ENC RECORD : [CLS] { " edward " : " s. ", " ruthazer " : { " born " : " in 1966 " }, " in " : " new york, ", " ny " : [ " ) ", 757326, null, " is " ], " canadian " : { " neuroscientist " : { " and " : null, " professor " : [ " at mcgill university ", 263844, 524301 ], " in " : " montreal, quebec " } } } [SEP]
  PROMPT     : what is the value at path canadian. neuroscientist. and? return value only.
  TARGET     : null [SEP]
  needle in context: True

--- sample 1 ---
  ENC RECORD : [CLS] { " choanephora " : " cucurbitarum is a ", " fungal " : [ " plant pathogen that " ], " causes " : " fruit and blossom ", " rot " : " of various cucurbits ", " it " : { " can " : { " also " : 227533 } } } [SEP]
  PROMPT     : key path : rot. return value only.
  TARGET     : of various cucurbits [SEP]
  needle in context: True

--- sample 2 ---
  

## 4. `jsonata` — JSONata/jq selection + transform

In [9]:
jsonata_cfg = JsonataRecallCfg(
    min_fields=4, max_fields=10, max_depth=3, max_array_len=5,
    value_max_words=3, transform_prob=0.35,
)
jsonata_ds = JsonataRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=jsonata_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
jsonata_ds.shuffle(seed=seed)
jsonata_batch = next(create_jsonata_recall_dataloader(jsonata_ds, batch_size=batch_size))
show_batch('jsonata', jsonata_batch)

R0. Create JsonataRecallDataset dataloader. batch_size=5.
### jsonata  |  batch_size=5  enc_len=86  dec_prompt_len=19  dec_cite_len=6

--- sample 0 ---
  ENC RECORD : [CLS] { " edward " : " s. ", " ruthazer " : { " born " : " in 1966 " }, " in " : [ " new york, ", " ny ) " ], " is " : 372733, " canadian " : " neuroscientist " } [SEP]
  PROMPT     : [ tier - e ] jq :. canadian. return value only.
  TARGET     : neuroscientist [SEP]
  needle in context: True

--- sample 1 ---
  ENC RECORD : [CLS] { " choanephora " : " cucurbitarum ", " is " : { " fungal " : " plant ", " pathogen " : [ null, " that ", " causes fruit and " ], " blossom " : " rot of various ", " cucurbits " : [ ". it can " ], " also " : " affect okra, " } } [SEP]
  PROMPT     : [ tier - c ] jq :. is. pathogen | length. return value only.
  TARGET     : 3 [SEP]
  needle in context: False

--- sample 2 ---
  ENC RECORD : [CLS] { " francisco " : 157273, " escudero " : " may refer to ", " casquino " : 371942, " peruvian " : " p

## 5. `xmlxpath` — XML/XPath → node value

In [10]:
xmlxpath_cfg = XmlXpathRecallCfg(
    min_nodes=4, max_nodes=12, max_depth=4, max_children=4, value_max_words=3,
)
xmlxpath_ds = XmlXpathRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=xmlxpath_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
xmlxpath_ds.shuffle(seed=seed)
xmlxpath_batch = next(create_xml_xpath_recall_dataloader(xmlxpath_ds, batch_size=batch_size))
show_batch('xmlxpath', xmlxpath_batch)

R0. Create XmlXpathRecallDataset dataloader. batch_size=5.
### xmlxpath  |  batch_size=5  enc_len=88  dec_prompt_len=21  dec_cite_len=8

--- sample 0 ---
  ENC RECORD : [CLS] < edward > s. < / edward > [SEP]
  PROMPT     : xml query : extract / edward / text ( ). return value only.
  TARGET     : s. [SEP]
  needle in context: True

--- sample 1 ---
  ENC RECORD : [CLS] < choanephora > cucurbitarum is a < / choanephora > [SEP]
  PROMPT     : xml query : extract / choanephora / text ( ). return value only.
  TARGET     : cucurbitarum is a [SEP]
  needle in context: True

--- sample 2 ---
  ENC RECORD : [CLS] < francisco > escudero may < / francisco > [SEP]
  PROMPT     : xml query : extract / francisco / text ( ). return value only.
  TARGET     : escudero may [SEP]
  needle in context: True


## 6. `sql` — SQL select / aggregate

In [11]:
sql_cfg = SqlSelectRecallCfg(
    min_rows=4, max_rows=8, min_cols=3, max_cols=5,
    value_max_words=3, transform_prob=0.30,
)
sql_ds = SqlSelectRecallDataset(
    ds, tkz_enc, max_seq_len=max_seq_len, n_special_toks=n_special_toks,
    cfg=sql_cfg, device=device, tkz_dec=tkz_dec, seed=seed,
)
sql_ds.shuffle(seed=seed)
sql_batch = next(create_sql_select_recall_dataloader(sql_ds, batch_size=batch_size))
show_batch('sql', sql_batch)

R0. Create SqlSelectRecallDataset dataloader. batch_size=5.
### sql  |  batch_size=5  enc_len=192  dec_prompt_len=24  dec_cite_len=5

--- sample 0 ---
  ENC RECORD : [CLS] | id | edward | ruthazer | born | | - - - | - - - | - - - | - - - | | 1 | in 1966 in | new york, | ny | | 2 | ) is a | canadian neuroscientist and | professor at mcgill | | 3 | university in | montreal, quebec |. | | 4 | research ruthazer | ' s research | utilizes in vivo | | 5 | multiphoton fluorescence | microscopy of | developing brain cells | | 6 | in conjunction with | patterned visual stimulation | and electrophysiological | | 7 | recordings to understand | how patterned | sensory experience | | 8 | impacts the development | and refinement | of | [SEP]
  PROMPT     : [ tier - c ] sql : select count ( * ) from t where id < = 4 ; return value only.
  TARGET     : 4 [SEP]
  needle in context: True

--- sample 1 ---
  ENC RECORD : [CLS] | id | choanephora | cucurbitarum | | - - - | - - - | - - - | | 1 | 62364 | is 

## Summary check

Confirm across every dataset that the decoder target value is recoverable from the
encoded context for all samples in the batch (the property that forces the model to
read the context embeddings rather than rely on priors).

In [26]:
all_batches = {
    'cite': cite_batch,
    'keyval': keyval_batch,
    'jsonfield': jsonfield_batch,
    'jsonata': jsonata_batch,
    'xmlxpath': xmlxpath_batch,
    'sql': sql_batch,
}
print(f'{"dataset":<12} {"samples":>8} {"needle_in_ctx":>14}')
print('-' * 36)
for name, b in all_batches.items():
    bs = b.inp_toks.shape[0]
    ok = sum(
        _contains_subseq(list(s.toks_inp), list(s.toks_cite))
        for s in b.tokens_subsets
    )
    print(f'{name:<12} {bs:>8} {f"{ok}/{bs}":>14}')

dataset       samples  needle_in_ctx
------------------------------------
cite                5            5/5
keyval              5            5/5
jsonfield           5            5/5
jsonata             5            4/5
xmlxpath            5            5/5
sql                 5            5/5
